# Quick start

This notebook computes a Fisher matrix for an EMRI with StableEMRIFisher (SEF) in three steps:

1. Set up `StableEMRIFisher` with a FEW waveform model.
2. Call it with the source parameters to get the Fisher matrix.
3. Invert the Fisher matrix to get the parameter uncertainties.

To keep it short and runnable on a laptop, it uses the Schwarzschild eccentric flux model without the LISA response, with a sky-averaged LISA noise curve and a 0.1-year observation.

In [1]:
import numpy as np

import few
from few.waveform import FastSchwarzschildEccentricFlux

from stableemrifisher.fisher import StableEMRIFisher

use_gpu = False  # set to True to run on an NVIDIA GPU

if not use_gpu:
    # restrict FEW to its CPU backend
    cfg_set = few.get_config_setter(reset=True)
    cfg_set.enable_backends("cpu")

## 1. Set up StableEMRIFisher

`StableEMRIFisher` takes the FEW waveform class (not an instance) and its keyword arguments, the observation time `T` (years) and the sampling interval `dt` (seconds). Without a `ResponseWrapper` or `noise_model`, SEF uses the sky-averaged LISA sensitivity curve on the two polarisations $h_+$ and $h_\times$.

`der_order` is the order of the finite-difference stencils SEF uses for the derivatives of the waveform.

In [2]:
sef = StableEMRIFisher(
    waveform_class=FastSchwarzschildEccentricFlux,
    waveform_class_kwargs=dict(
        inspiral_kwargs=dict(err=1e-11),  # trajectory integrator tolerance
        mode_selector_kwargs=dict(mode_selection_threshold=1e-5),
    ),
    T=0.1,  # observation time, years
    dt=10.0,  # sampling interval, seconds
    der_order=4,
    use_gpu=use_gpu,
)

No noise model or response wrapper provided.
Defaulting to the sky-averaged sensitivity curve


## 2. Compute the Fisher matrix

The source parameters are given as a dictionary, in the order FEW expects them. `param_names` selects the parameters of the Fisher matrix. The spin `a`, inclination `xI0` and polar phase `Phi_theta0` are left out because this model is for a non-spinning black hole and equatorial orbits. The sky and spin angles are left out too: without the LISA response, the waveform depends on the four angles only through two combinations, so they cannot all be measured.

For each parameter, SEF first searches for a stable finite-difference step size, then computes the derivatives of the waveform and the Fisher matrix.

In [3]:
wave_params = dict(
    m1=1e6,  # primary mass, solar masses
    m2=10.0,  # secondary mass, solar masses
    a=0.0,  # primary spin
    p0=9.5,  # initial semi-latus rectum
    e0=0.4,  # initial eccentricity
    xI0=1.0,  # cosine of the initial inclination
    dist=0.5,  # luminosity distance, Gpc
    qS=np.pi / 3,  # sky location, polar angle
    phiS=np.pi / 4,  # sky location, azimuthal angle
    qK=np.pi / 6,  # spin orientation, polar angle
    phiK=np.pi / 8,  # spin orientation, azimuthal angle
    Phi_phi0=1.0,  # initial azimuthal phase
    Phi_theta0=0.0,  # initial polar phase
    Phi_r0=0.0,  # initial radial phase
)

param_names = ["m1", "m2", "p0", "e0", "dist", "Phi_phi0", "Phi_r0"]

Fisher = sef(wave_params, param_names=param_names)

print("Fisher matrix shape:", Fisher.shape)

Waveform Generated. SNR: 39.08183361445396
Computing Fisher   : 100%|██████████| 7/7 [00:07<00:00,  1.09s/deriv]          
Calculated Fisher is positive-definite.
Fisher matrix shape: (7, 7)


The step size chosen for each parameter is stored in `sef.deltas` (0 means the derivative is computed analytically).

In [4]:
sef.deltas

{'m1': 19.306977288832496,
 'm2': 0.00019306977288832496,
 'p0': 0.00018341628424390865,
 'e0': 7.722790915533e-06,
 'dist': 0.0,
 'Phi_phi0': 0.0,
 'Phi_r0': 0.0}

## 3. Parameter uncertainties

The inverse of the Fisher matrix approximates the covariance matrix of the parameters, so the square roots of its diagonal are the $1\sigma$ uncertainties.

In [5]:
covariance = np.linalg.inv(Fisher)
sigma = np.sqrt(np.diag(covariance))

for name, s in zip(param_names, sigma):
    print(f"{name:9s} = {wave_params[name]:.6g} +/- {s:.3g}")

m1        = 1e+06 +/- 2.5e+03
m2        = 10 +/- 0.09
p0        = 9.5 +/- 0.000891
e0        = 0.4 +/- 0.00233
dist      = 0.5 +/- 0.0147
Phi_phi0  = 1 +/- 0.0739
Phi_r0    = 0 +/- 0.054


## Next steps

- `fisher_matrix_no_response.ipynb`: the full Fisher matrix without the response, including the sky angles, and how SEF reports degenerate parameters.
- `fisher_matrix_with_response.ipynb`: the full Fisher matrix with the LISA response and TDI noise, and covariance ellipses.
- `custom_psd_tutorial.ipynb`: using your own noise model.
- `5PNAAK_direct_derivative.ipynb`: a waveform model without a stable derivative, using direct finite differences.
- `stable_derivative_one_param_no_response.ipynb` and `stable_derivative_one_param_with_response.ipynb`: checking a Fisher estimate against the true likelihood.
- `derivatives_tutorial.ipynb`: using the derivative classes directly, and validating the stable derivative.
- `inner_product.ipynb`: SEF's inner product and PSD utilities on their own.